In [0]:
import json
import requests
import time
from pathlib import Path

STATIONS_FILE = Path("/Volumes/weather/raw/ana_volume/estaciones_rio_uruguai_pluvio_fluvio.json")

OUTPUT_DIR = Path("/Volumes/weather/raw/ana_volume/historical")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BASE_URL = "http://www.snirh.gov.br/hidroweb/rest/api/seriehistorica"

TIPO = 3  # 1=cota 2=vazao 3=chuva
DELAY = 0.3


def load_stations():

    estaciones = json.loads(STATIONS_FILE.read_text())

    codigos = []

    for e in estaciones:

        codigo = e.get("codigoestacao")

        if codigo:
            codigos.append(codigo)

    return codigos


def download_station(codigo):

    output = OUTPUT_DIR / f"{codigo}.csv"

    if output.exists():
        print(f"skip {codigo}")
        return

    url = f"{BASE_URL}?codEstacao={codigo}&tipo={TIPO}&formato=csv"

    try:

        r = requests.get(url, timeout=120)

        if r.status_code != 200:
            print(f"HTTP {r.status_code} estacion {codigo}")
            print(r.text[:200])
            return

        if len(r.text) < 50:
            print(f"sin datos {codigo}")
            return

        output.write_text(r.text)

        print(f"ok {codigo}")

    except Exception as e:

        print(f"error {codigo} {e}")


stations = load_stations()

print("estaciones:", len(stations))

for i, codigo in enumerate(stations):

    print(f"{i+1}/{len(stations)}", end=" ")

    download_station(codigo)

    time.sleep(DELAY)